# Transaction Graph Analysis

This notebook covers temporal and structural analysis of Stellar transaction graphs:

1. **Graph snapshots** — rolling time-window graphs
2. **Structural importance** — PageRank, betweenness, clustering
3. **Temporal decay features** — recency-weighted activity
4. **Asset diversity** — multi-asset account profiling
5. **Graph validation** — data quality checks before training

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## 1. Graph Snapshots

AstroML builds rolling time-window snapshots of the transaction graph.
Each snapshot captures activity within a configurable window (e.g. 30 days).

In [ ]:
# Equivalent CLI:
# python -m astroml.graph.build_snapshot --window 30d

from astroml.features.graph.snapshot import GraphSnapshot

# Generate synthetic time-series transactions
rng = np.random.default_rng(42)
accounts = [f'G{i:04d}' for i in range(20)]
base_time = datetime(2024, 1, 1)

transactions = []
for i in range(300):
    src, dst = rng.choice(accounts, size=2, replace=False)
    transactions.append({
        'source': src,
        'destination': dst,
        'amount': float(rng.lognormal(3, 1)),
        'asset': rng.choice(['XLM', 'USDC', 'BTC']),
        'timestamp': (base_time + timedelta(days=int(rng.uniform(0, 60)))).isoformat(),
    })

snapshot = GraphSnapshot(window_days=30)
G_snap = snapshot.build(transactions, reference_date='2024-02-01')

print(f'30-day snapshot: {G_snap.number_of_nodes()} nodes, {G_snap.number_of_edges()} edges')

## 2. Structural Importance Features

Compute graph-theoretic features that capture each account's role in the network.

In [ ]:
from astroml.features.structural_importance import compute_structural_features

struct_features = compute_structural_features(G_snap)

# Show top accounts by PageRank
pageranks = {node: feats['pagerank'] for node, feats in struct_features.items()}
top_accounts = sorted(pageranks, key=pageranks.get, reverse=True)[:5]

print('Top 5 accounts by PageRank:')
for acc in top_accounts:
    f = struct_features[acc]
    print(f'  {acc}: pagerank={f["pagerank"]:.4f}, '
          f'in_degree={f["in_degree"]}, out_degree={f["out_degree"]}')

In [ ]:
# Visualize degree distribution
degrees = [d for _, d in G_snap.degree()]

plt.figure(figsize=(8, 4))
plt.hist(degrees, bins=15, color='#4299e1', edgecolor='white')
plt.xlabel('Degree')
plt.ylabel('Count')
plt.title('Account Degree Distribution (30-day snapshot)')
plt.tight_layout()
plt.show()

## 3. Temporal Decay Features

Recent activity is weighted more heavily than older activity using exponential decay.

In [ ]:
from astroml.features.temporal_decay import TemporalDecayFeatures

tdf = TemporalDecayFeatures(half_life_days=7)
decay_features = tdf.compute(transactions, reference_date='2024-02-01')

# Show features for a sample account
sample_account = accounts[0]
if sample_account in decay_features:
    print(f'Temporal decay features for {sample_account}:')
    for k, v in decay_features[sample_account].items():
        print(f'  {k}: {v:.4f}')
else:
    print(f'{sample_account} had no transactions in window')

## 4. Asset Diversity Profiling

Accounts that transact in many different assets may indicate market makers or
sophisticated actors — useful signal for both fraud detection and clustering.

In [ ]:
from astroml.features.asset_diversity import compute_asset_diversity

diversity = compute_asset_diversity(transactions)

# Distribution of asset diversity scores
scores = [v['diversity_score'] for v in diversity.values()]

plt.figure(figsize=(7, 4))
plt.hist(scores, bins=10, color='#68d391', edgecolor='white')
plt.xlabel('Asset Diversity Score')
plt.ylabel('Accounts')
plt.title('Asset Diversity Distribution')
plt.tight_layout()
plt.show()

print(f'Mean diversity: {np.mean(scores):.3f}')
print(f'Max diversity:  {np.max(scores):.3f}')

## 5. Graph Validation

Before training, validate the graph for data quality issues.

In [ ]:
from astroml.features.graph_validation import GraphValidator

validator = GraphValidator()
report = validator.validate(G_snap)

print('Graph validation report:')
print(f'  Nodes:              {report["num_nodes"]}')
print(f'  Edges:              {report["num_edges"]}')
print(f'  Self-loops:         {report["self_loops"]}')
print(f'  Isolated nodes:     {report["isolated_nodes"]}')
print(f'  Weakly connected:   {report["is_weakly_connected"]}')
print(f'  Largest component:  {report["largest_component_size"]} nodes')
print(f'  Warnings:           {len(report["warnings"])}')
for w in report['warnings']:
    print(f'    - {w}')

## 6. Putting It All Together: Feature Matrix for ML

Combine all features into a single matrix ready for GNN training.

In [ ]:
from astroml.features.node_features import compute_node_features

# Merge all feature sources
all_features = compute_node_features(
    G_snap,
    include_structural=True,
    include_temporal=True,
    transactions=transactions,
    reference_date='2024-02-01',
)

nodes = list(all_features.keys())
X = np.array([list(all_features[n].values()) for n in nodes])

print(f'Feature matrix shape: {X.shape}')
print(f'Features per node:    {X.shape[1]}')
print(f'Feature names:        {list(list(all_features.values())[0].keys())}')

## Summary

You now have a complete feature pipeline:

```
Raw ledger → Snapshot → Structural + Temporal + Asset features → Feature matrix → GNN
```

See `docs/experiment-configs.md` for how to configure full training runs with Hydra.